# Week 2 Exercise: Technical Tutor Pro

This builds on the [Week 1 exercise](../../week1/week1%20EXERCISE.ipynb) (a technical Q&A tool using
GPT-4o-mini and a local Ollama model) and turns it into the full prototype the Week 2 exercise asks for:

- **Gradio UI** (`gr.Blocks`, following the Day 5 pattern)
- **Streaming** responses, token by token
- A **system prompt** that gives the assistant a technical-tutor persona
- The ability to **switch between models** (OpenAI, Claude, and a local Ollama model)
- **Bonus: tool use** — the tutor can call a `lookup_glossary_term` tool so its answers use this
  course's own definitions of terms like "RAG" or "temperature" instead of generic ones
- **Bonus: voice in / voice out** — ask by microphone (Whisper transcription) and optionally have the
  answer read back (OpenAI TTS)

Set `RUN_LOCAL_MODEL = False` below if you don't have Ollama running, and leave `ANTHROPIC_API_KEY`
unset if you don't want the Claude option — both are detected automatically and simply won't appear
in the model dropdown.

In [1]:
# imports

import os
import json
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

## Setting up

Same pattern as Week 1: load keys from `.env`, and only enable a model if we can actually reach it.

In [2]:
# set up environment

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

RUN_LOCAL_MODEL = True  # set False to skip Ollama entirely
OLLAMA_BASE_URL = "http://localhost:11434/v1"
LOCAL_MAX_TOKENS = 200  # local CPU models can be slow - keep answers short, as in the Week 1 exercise

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set - the Claude option will be hidden")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [3]:
# constants

MODEL_GPT = "gpt-4o-mini"
MODEL_CLAUDE = "claude-sonnet-4-5-20250929"
MODEL_OLLAMA = "qwen3.5:latest"  # swap for whatever you have pulled, e.g. llama3.2

system_message = """
You are a friendly, precise technical tutor. You explain code and technical concepts clearly and
concisely to a learner who is following an AI engineering course. Use markdown, and include short
code examples in code blocks whenever they'd help.

If the learner's question involves a term that might be specific to this course (for example an
LLM engineering concept like RAG, embeddings, temperature, tokens, agents, or fine-tuning), use the
lookup_glossary_term tool to check this course's own definition before answering, so your explanation
lines up with the terminology the learner has already seen.
"""

## Connecting to the models

We reuse the trick from Day 2: point the `OpenAI` client at Anthropic's OpenAI-compatible endpoint so Claude can be called with the same `chat.completions` interface, and do the same for the local Ollama server.

In [4]:
# clients - only create the ones we actually have credentials/access for

openai_client = OpenAI() if openai_api_key else None

claude_client = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/"
) if anthropic_api_key else None

ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama") if RUN_LOCAL_MODEL else None

# Model registry for the dropdown: label -> (client, model name, whether it supports our tool)
MODELS = {}
if openai_client:
    MODELS["GPT-4o-mini"] = (openai_client, MODEL_GPT, True)
if claude_client:
    MODELS["Claude Sonnet"] = (claude_client, MODEL_CLAUDE, True)
if ollama_client:
    MODELS[f"Ollama ({MODEL_OLLAMA}, local)"] = (ollama_client, MODEL_OLLAMA, False)

if not MODELS:
    raise RuntimeError("No models available - set an API key or start Ollama")

print("Available models:", list(MODELS.keys()))

Available models: ['GPT-4o-mini', 'Claude Sonnet', 'Ollama (qwen3.5:latest, local)']


## Bonus: a glossary tool

A small, course-specific glossary the tutor can call as a tool. This is deliberately simple - the point is to demonstrate the tool-calling loop from Day 4/5, not to build a real knowledge base.

In [5]:
# the glossary the tool looks up

GLOSSARY = {
    "token": "A chunk of text (often a word-piece) that an LLM reads or generates one at a time.",
    "temperature": "A generation setting that controls randomness: near 0 is deterministic and focused, "
                   "higher values (e.g. 0.7-1.0) produce more varied, creative output.",
    "embedding": "A list of numbers (a vector) that represents the meaning of a piece of text, so that "
                 "similar meanings end up as nearby vectors.",
    "rag": "Retrieval-Augmented Generation: looking up relevant chunks of external text and inserting them "
           "into the prompt so the LLM can answer using information it wasn't trained on.",
    "agent": "An LLM that can autonomously decide which tools to call, in what order, to complete a task, "
             "rather than just producing a single reply.",
    "system prompt": "An instruction given to an LLM before the conversation starts, used to set its "
                      "persona, tone, or rules of behaviour for the rest of the chat.",
    "context window": "The maximum number of tokens (prompt + conversation history + generated reply) an "
                       "LLM can take into account at once.",
    "fine-tuning": "Further training an existing model on a smaller, specific dataset so it adapts its "
                   "behaviour or knowledge for a particular task.",
}

def lookup_glossary_term(term):
    print(f"TOOL CALLED: lookup_glossary_term({term!r})", flush=True)
    definition = GLOSSARY.get(term.strip().lower())
    if definition:
        return f"{term}: {definition}"
    return f"No glossary entry for '{term}'. Known terms: {', '.join(GLOSSARY.keys())}"

lookup_glossary_term("RAG")

TOOL CALLED: lookup_glossary_term('RAG')


"RAG: Retrieval-Augmented Generation: looking up relevant chunks of external text and inserting them into the prompt so the LLM can answer using information it wasn't trained on."

In [6]:
# the tool's JSON schema, and the tool-call handler

glossary_function = {
    "name": "lookup_glossary_term",
    "description": "Look up this course's own definition of an AI/LLM engineering term, e.g. 'RAG' or "
                   "'temperature', so explanations use consistent terminology.",
    "parameters": {
        "type": "object",
        "properties": {
            "term": {
                "type": "string",
                "description": "The technical term to look up, e.g. 'embedding'"
            }
        },
        "required": ["term"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": glossary_function}]


def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "lookup_glossary_term":
            arguments = json.loads(tool_call.function.arguments)
            result = lookup_glossary_term(arguments.get("term", ""))
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    return responses

## Bonus: voice in, voice out

Whisper turns a recorded question into text; OpenAI TTS turns the final answer back into speech. Both are optional - the app works with typed text and no audio output too.

In [7]:
# speech-to-text and text-to-speech, both via OpenAI - skipped gracefully if there's no OpenAI key

def transcribe_audio(audio_filepath):
    if audio_filepath is None or openai_client is None:
        return gr.update()
    with open(audio_filepath, "rb") as f:
        transcript = openai_client.audio.transcriptions.create(model="whisper-1", file=f)
    return transcript.text


def talker(text):
    if not text or openai_client is None:
        return None
    response = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=text
    )
    fd, path = tempfile.mkstemp(suffix=".mp3")
    with os.fdopen(fd, "wb") as f:
        f.write(response.content)
    return path

## Putting it together: streaming + model switching + tools

Tool calls aren't compatible with streaming in the OpenAI API, so we resolve any tool calls first with a normal (non-streamed) request, then stream the final answer once we know no more tools are needed. The local Ollama model skips the tool step entirely and is capped at `LOCAL_MAX_TOKENS` to stay responsive on CPU, exactly as in the Week 1 exercise.

In [8]:
def chat(history, model_choice, read_aloud):
    client, model, supports_tools = MODELS[model_choice]
    messages = [{"role": "system", "content": system_message}] + history

    if supports_tools:
        response = client.chat.completions.create(model=model, messages=messages, tools=tools)
        while response.choices[0].finish_reason == "tool_calls":
            tool_message = response.choices[0].message
            messages.append(tool_message)
            messages.extend(handle_tool_calls(tool_message))
            response = client.chat.completions.create(model=model, messages=messages, tools=tools)
        stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    else:
        stream = client.chat.completions.create(
            model=model, messages=messages, stream=True, max_tokens=LOCAL_MAX_TOKENS
        )

    history = history + [{"role": "assistant", "content": ""}]
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        if delta:
            history[-1]["content"] += delta
            yield history, None

    if read_aloud:
        yield history, talker(history[-1]["content"])

## The Gradio UI

In [ ]:
def user_turn(user_message, history):
    if not user_message:
        return gr.update(), history
    return "", history + [{"role": "user", "content": user_message}]


with gr.Blocks(title="Technical Tutor Pro") as ui:
    gr.Markdown("# \U0001F393 Technical Tutor Pro")
    gr.Markdown("Ask a technical question by text or voice, pick a model, and optionally hear the answer read back.")

    with gr.Row():
        model_choice = gr.Dropdown(choices=list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model")
        read_aloud = gr.Checkbox(label="Read the answer aloud", value=False)

    chatbot = gr.Chatbot(height=450, type="messages", label="Technical Tutor")

    with gr.Row():
        message = gr.Textbox(
            label="Ask a technical question",
            placeholder="e.g. What does 'yield from' do in Python?",
            scale=4
        )
        audio_in = gr.Audio(sources=["microphone"], type="filepath", label="...or ask by voice", scale=1)

    with gr.Row():
        submit_btn = gr.Button("Ask", variant="primary")
        clear_btn = gr.ClearButton([message, chatbot])

    audio_out = gr.Audio(label="Spoken answer", autoplay=True)

    audio_in.stop_recording(transcribe_audio, inputs=audio_in, outputs=message)

    message.submit(user_turn, [message, chatbot], [message, chatbot]).then(
        chat, [chatbot, model_choice, read_aloud], [chatbot, audio_out]
    )
    submit_btn.click(user_turn, [message, chatbot], [message, chatbot]).then(
        chat, [chatbot, model_choice, read_aloud], [chatbot, audio_out]
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


TOOL CALLED: lookup_glossary_term('RAG')
TOOL CALLED: lookup_glossary_term('Hugging Face')


## Where to take this further

- Add more glossary terms, or swap the glossary tool for a real lookup (e.g. searching the course's own notebooks)
- Add a second tool, so the model has to choose between them
- Persist the glossary lookups so repeated questions about the same term are instant
- Try `gr.ChatInterface` instead of `gr.Blocks` for a simpler UI once you drop the voice/model-switch extras